In [76]:
import pandas as pd
import time

start = time.time()
file_path = r"C:\Users\Mahad\Downloads\Crimes_-_2001_to_Present_20260211.csv"

df = pd.read_csv(
    file_path,
    usecols=["Date", "Primary Type","Description","Location Description", "Arrest", "District", "Ward", "Community Area", "Domestic"],
    parse_dates=["Date"],
    date_format="%m/%d/%Y %I:%M:%S %p"
)


In [74]:
df.head()

,Date,Primary Type,Arrest,Domestic,District,Ward,Community Area
0,2026-02-02,BURGLARY,False,False,4.0,10.0,55.0
1,2026-02-02,BATTERY,False,True,18.0,42.0,8.0
2,2026-02-02,CRIMINAL DAMAGE,False,False,19.0,47.0,5.0
3,2026-02-02,DECEPTIVE PRACTICE,False,False,18.0,27.0,8.0
4,2026-02-02,CRIMINAL TRESPASS,False,False,18.0,42.0,8.0


In [69]:
df = df.dropna()

In [70]:
df["incident_hour"] = df["Date"].dt.hour
df["incident_day"] = df["Date"].dt.day_name()
df["incident_month"] = df["Date"].dt.month
df["incident_year"] = df["Date"].dt.year

In [71]:
df["District"] = df["District"].astype("Int64")
df["Ward"] = df["Ward"].astype("Int64")
df["Community Area"] = df["Community Area"].astype("Int64")

KeyError: 'District'

In [ ]:
def recode_crime(x):
    if x in ["HOMICIDE", "ASSAULT", "BATTERY", "ROBBERY",
             "KIDNAPPING", "INTIMIDATION", "STALKING"]:
        return "VIOLENT CRIME"
    elif x in ["CRIM SEXUAL ASSAULT", "CRIMINAL SEXUAL ASSAULT",
               "SEX OFFENSE", "HUMAN TRAFFICKING"]:
        return "SEXUAL / TRAFFICKING"
    elif x in ["THEFT", "BURGLARY", "MOTOR VEHICLE THEFT",
               "CRIMINAL DAMAGE", "CRIMINAL TRESPASS",
               "DECEPTIVE PRACTICE", "ARSON"]:
        return "PROPERTY CRIME"
    elif x in ["NARCOTICS", "OTHER NARCOTIC VIOLATION"]:
        return "DRUG OFFENSE"
    elif x in ["PROSTITUTION", "GAMBLING", "LIQUOR LAW VIOLATION",
               "PUBLIC INDECENCY", "OBSCENITY", "RITUALISM",
               "PUBLIC PEACE VIOLATION"]:
        return "PUBLIC ORDER"
    elif x in ["WEAPONS VIOLATION", "CONCEALED CARRY LICENSE VIOLATION"]:
        return "WEAPONS OFFENSE"
    elif x in ["OFFENSE INVOLVING CHILDREN"]:
        return "CRIMES AGAINST CHILDREN"
    elif x in ["INTERFERENCE WITH PUBLIC OFFICER"]:
        return "LAW ENFORCEMENT RELATED"
    elif x in ["OTHER OFFENSE"]:
        return "MISCELLANEOUS CRIME"
    elif x in ["NON-CRIMINAL"]:
        return "NON-CRIMINAL"
    else:
        return "UNCLASSIFIED"

df["Crime Category"] = df["Primary Type"].apply(recode_crime)

Aggregate per hour

In [ ]:
hourly_counts = df.groupby(["Crime Category", "incident_hour"]) \
                  .size() \
                  .unstack(fill_value=0)

 Crime arrest rate

In [ ]:
crime_rate = df.groupby("Crime Category")["Arrest"] \
               .mean() \
               .reset_index(name="arrest_rate")

Domestic crime rate

In [ ]:
domestic_counts = df.groupby(["Crime Category", "Domestic"]) \
                    .size() \
                    .unstack(fill_value=0)

domestic_counts["domestic_rate"] = domestic_counts[True] / (domestic_counts[True] + domestic_counts[False])



## Extract 2021 data separately

In [72]:
df_2021 = df[df["incident_year"] == 2021].copy()
df_2021.to_csv("Crimes_2021.csv", index=False)
print("Rows in 2021:", len(df_2021))

PermissionError: [Errno 13] Permission denied: 'Crimes_2021.csv'

In [ ]:
end = time.time()
ram = df.memory_usage(deep=True).sum() / 1024**2

In [ ]:
print("Execution time:", round(end - start, 2), "seconds")
print("Memory Usage:", round(ram, 2), "MB")